# LangChain Complete Guide: From Fundamentals to Advanced RAG

This notebook covers:
1. **Setup & Configuration** - Dependencies and OpenRouter/OpenAI setup
2. **LangChain Fundamentals** - LLM calls, Prompts, Output Parsers
3. **Memory Systems** - Buffer, Window, Summary, Entity Memory
4. **Chains** - LCEL, Sequential, Router chains


---

## Section 1: Setup & Configuration

In [46]:
# Install required dependencies
!pip install -q \
    langchain>=0.3.0 \
    langchain-openai>=0.2.0 \
    langchain-community>=0.3.0 \
    langchain-core>=0.3.0 \
    chromadb>=0.5.0 \
    pypdf>=4.0 \
    python-dotenv>=1.0 \
    pandas>=2.0 \
    tiktoken>=0.7.0 \
    sentence-transformers>=3.0 \
    gradio>=4.0 \
    fpdf2>=2.7.0

zsh:1: 0.3.0 not found


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ============================================================
# CONFIGURATION: Choose ONE of the options below
# ============================================================

# OPTION 1: OpenRouter (supports multiple models - GPT, Claude, etc.)
# Set OPENROUTER_API_KEY in your .env file
USE_OPENROUTER = True  # Set to False for direct OpenAI

if USE_OPENROUTER:
    API_KEY = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
    BASE_URL = "https://openrouter.ai/api/v1"
    MODEL_NAME = "openai/gpt-4o-mini"  # or "anthropic/claude-3.5-sonnet"
    print("Using OpenRouter API")
else:
    # OPTION 2: Direct OpenAI
    # Set OPENAI_API_KEY in your .env file
    API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
    BASE_URL = None  # Uses default OpenAI endpoint
    MODEL_NAME = "gpt-4o-mini"  # or "gpt-4o", "gpt-3.5-turbo"
    print("Using OpenAI API directly")

print("Configuration loaded successfully!")

Using OpenRouter API
Configuration loaded successfully!


In [ ]:
from langchain_openai import ChatOpenAI, ChatAnthropic

# Initialize the LLM (works with both OpenRouter and OpenAI)
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,  # None for OpenAI, URL for OpenRouter
    temperature=0.7,
    max_tokens=1000,
    tiktoken_model_name="gpt-4o-mini",  # for token counting (ConversationSummaryBufferMemory)
)

llm1= ChatAnthropic(
    model="claude-3-5-sonnet-20240620",
    anthropic_api_key=API_KEY,
    anthropic_api_base=BASE_URL,  # None for OpenAI, URL for OpenRouter
    temperature=0.7,
    max_tokens=1000,
)

# Test the connection
response = llm.invoke("Say 'Hello, LangChain!' in a creative way.")
print(response.content)

Greetings, oh wondrous tapestry of LangChain! 🌟 May our dialogue weave together threads of knowledge and innovation!


---
## Section 2: LangChain Fundamentals

### 2.1 LLM Calls

In [51]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Method 1: Simple string invocation
response = llm.invoke("What is the capital of France?")
print("Simple invocation:")
print(response.content)
print()

Simple invocation:
The capital of France is Paris.



In [52]:
# Method 2: Using message objects for more control
messages = [
    SystemMessage(content="You are a helpful geography expert. Be concise."),
    HumanMessage(content="What is the capital of France?")
]

response = llm.invoke(messages)
print("With system message:")
print(response.content)

With system message:
The capital of France is Paris.


In [53]:
# Method 3: Batch processing multiple inputs
from typing import Any


questions = [
    "What is 2 + 2?",
    "What color is the sky?",
    "Who wrote Romeo and Juliet?"
]

responses = llm.batch(questions)
print("Batch processing:")
for q, r in zip(questions, responses):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

Batch processing:
Q: What is 2 + 2?
A: 2 + 2 equals 4.

Q: What color is the sky?
A: The color of the sky can vary depending on several factors, including time of day, weather conditions, and atmospheric particles. During a clear day, the sky typically appears blue due to the scattering of sunlight by the Earth’s atmosphere. This scattering causes shorter blue wavelengths of light to be more prominent. At sunrise and sunset, the sky can take on shades of red, orange, and pink as the sunlight passes through more of the atmosphere. On cloudy or overcast days, the sky may appear gray. Additionally, at night, the sky can appear black or dark blue, dotted with stars.

Q: Who wrote Romeo and Juliet?
A: "Romeo and Juliet" was written by William Shakespeare. It is one of his most famous plays and was likely composed in the early 1590s.



In [59]:
# llm1.batch(questions)

In [ ]:
# for i in questions:
#     openai.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant."},
#             {"role": "user", "content": i}
#         ]
#     )
#     response.content.message[0].

In [ ]:
responses

[AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 7.05e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 7.05e-06, 'upstream_inference_prompt_cost': 2.25e-06, 'upstream_inference_completions_cost': 4.8e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'gen-1772342027-zxduycftmG5c8KIoX1BG', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ca7d1-1966-7290-9ec3-9832d8a95de7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [56]:
"Hi my name is {}".format("Divij")

'Hi my name is Divij'

### 2.2 Prompts & Prompt Templates

In [68]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate

# Why it failed: The template only has input_variables=["topic"], so there is no {system_prompt}
# in the template. Passing system_prompt to .format() does nothing - it gets ignored!
# Fix: Use ChatPromptTemplate so the rule goes in a proper system message (models follow it better).

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a finance expert. You can ONLY answer questions related to finance. If the question is not about finance, reply exactly: \"I'm sorry, I can't answer that question.\""),
    ("human", "Explain {topic} in simple terms for a beginner.")
])

messages = chat_template.format_messages(topic="machine learning")
print("Sending: system (finance-only rule) + user (topic='machine learning')")
print()

response = llm.invoke(messages)
print("Response:")
print(response.content)

Formatted prompt:
Explain Why sky is blue? in simple terms for a beginner.

Response:
The sky looks blue because of a process called scattering. Here’s a simple way to understand it:

1. **Sunlight and Colors**: Sunlight may look white to us, but it’s actually made up of many colors, like red, orange, yellow, green, blue, and violet. These colors can be seen in a rainbow.

2. **Earth’s Atmosphere**: The air around us is filled with tiny particles and gases. When sunlight enters the atmosphere, it hits these particles.

3. **Scattering of Light**: Blue light waves are shorter and scatter more than the other colors when they hit these particles. This means that blue light gets spread out in all directions.

4. **Seeing the Blue Sky**: Because blue light is scattered everywhere in the sky, we see it as blue when we look up.

So, in simple terms, the sky is blue because the shorter blue light waves scatter more than other colors, making the sky appear blue to our eyes!


In [58]:
formatted = simple_template.format(topic="finance")
print("Formatted prompt:")
print(formatted)
print()

# Use with LLM
response = llm.invoke(formatted)
print("Response:")
print(response.content)

Formatted prompt:
Explain finance in simple terms for a beginner.

Response:
Sure! Finance is all about managing money. Here are some key concepts to help you understand it better:

1. **Money Management**: This is the process of budgeting, saving, and investing your money wisely so you can meet your financial goals, whether that's buying a house, saving for retirement, or just making ends meet.

2. **Income**: This is the money you earn, which can come from your job, investments, or other sources. It's important because it's what you use to pay for your needs and wants.

3. **Expenses**: These are the costs that you incur, like rent, groceries, and bills. Keeping track of your expenses helps you understand where your money is going.

4. **Budgeting**: This is creating a plan for how to spend your income. A budget helps you allocate your money to different areas, ensuring that you can cover your expenses and save for the future.

5. **Saving**: This involves setting aside a portion of 

In [ ]:
# ChatPromptTemplate for chat models (recommended)
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Always respond in a {tone} manner."),
    ("human", "{question}")
])

# Format and invoke
messages = chat_template.format_messages(
    role="helpful coding assistant",
    tone="friendly and encouraging",
    question="How do I write a for loop in Python?"
)

response = llm.invoke(messages)
print(response.content)

Writing a for loop in Python is quite simple and straightforward! A for loop is used to iterate over a sequence (like a list, tuple, string, or range). Here’s a basic structure of a for loop:

```python
for item in iterable:
    # Do something with item
```

### Example 1: Looping through a list
Here's an example that demonstrates how to use a for loop to iterate through a list of numbers:

```python
numbers = [1, 2, 3, 4, 5]
for number in numbers:
    print(number)
```

### Example 2: Looping through a string
You can also loop through a string to access each character:

```python
my_string = "Hello"
for char in my_string:
    print(char)
```

### Example 3: Using `range()`
If you want to loop a specific number of times, you can use the `range()` function:

```python
for i in range(5):  # This will loop from 0 to 4
    print(i)
```

### Summary
- Replace `item` with whatever variable name you want to use.
- Replace `iterable` with the list, string, or range you're working with.
- The c

In [ ]:
# FewShotPromptTemplate - teaching the model with examples
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "fast", "output": "slow"},
]

example_template = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Give the opposite of each word.",
    suffix="Input: {word}\nOutput:",
    input_variables=["word"],
)

formatted = few_shot_prompt.format(word="bright")
print("Few-shot prompt:")
print(formatted)
print()

response = llm.invoke(formatted)
print("Model's answer:", response.content)

Few-shot prompt:
Give the opposite of each word.

Input: happy
Output: sad

Input: tall
Output: short

Input: fast
Output: slow

Input: bright
Output:

Model's answer: Output: dim


### 2.3 Output Parsers

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from pydantic import BaseModel, Field
from typing import List

# 1. StrOutputParser - extracts string content from response
str_parser = StrOutputParser()

# Chain: prompt -> llm -> parser
chain = chat_template | llm | str_parser

result = chain.invoke({
    "role": "Python expert",
    "tone": "concise",
    "question": "What is a list comprehension?"
})

print("StrOutputParser result:")
print(result)  # Now it's a string, not an AIMessage object
print(f"Type: {type(result)}")

StrOutputParser result:
A list comprehension is a concise way to create lists in Python. It consists of brackets containing an expression followed by a `for` clause, and can include optional `if` statements to filter items. 

Example:
```python
squared_numbers = [x**2 for x in range(10) if x % 2 == 0]
```
This creates a list of squares of even numbers from 0 to 9.
Type: <class 'str'>


In [ ]:
# 2. CommaSeparatedListOutputParser
list_parser = CommaSeparatedListOutputParser()

list_prompt = PromptTemplate(
    template="List 5 {category}. Respond with ONLY a comma-separated list, nothing else.",
    input_variables=["category"]
)

chain = list_prompt | llm | list_parser
result = chain.invoke({"category": "programming languages"})

print("CommaSeparatedListOutputParser result:")
print(result)
print(f"Type: {type(result)}")

CommaSeparatedListOutputParser result:
['Python', 'Java', 'C++', 'JavaScript', 'Ruby']
Type: <class 'list'>


In [ ]:
# 3. JsonOutputParser with Pydantic model
class Book(BaseModel):
    """Information about a book."""
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author of the book")
    year: int = Field(description="The year the book was published")
    genres: List[str] = Field(description="List of genres")

json_parser = JsonOutputParser(pydantic_object=Book)

json_prompt = PromptTemplate(
    template="""Generate information about a famous {genre} book.
    
{format_instructions}

Respond with ONLY the JSON, no additional text.""",
    input_variables=["genre"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

chain = json_prompt | llm | json_parser
result = chain.invoke({"genre": "science fiction"})

print("JsonOutputParser result:")
print(result)
print(f"Type: {type(result)}")
print(f"Title: {result['title']}")

JsonOutputParser result:
{'title': 'Dune', 'author': 'Frank Herbert', 'year': 1965, 'genres': ['Science Fiction', 'Adventure', 'Fantasy']}
Type: <class 'dict'>
Title: Dune


In [ ]:
# 4. PydanticOutputParser - returns actual Pydantic objects
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class MovieReview(BaseModel):
    """A movie review."""
    movie_title: str = Field(description="Title of the movie")
    rating: int = Field(description="Rating from 1-10")
    summary: str = Field(description="Brief summary of the review")
    recommended: bool = Field(description="Whether you'd recommend this movie")

pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

review_prompt = PromptTemplate(
    template="""Write a review for the movie "{movie}".
    
{format_instructions}""",
    input_variables=["movie"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

chain = review_prompt | llm | pydantic_parser
result = chain.invoke({"movie": "The Matrix"})

print("PydanticOutputParser result:")
print(f"Movie: {result.movie_title}")
print(f"Rating: {result.rating}/10")
print(f"Summary: {result.summary}")
print(f"Recommended: {result.recommended}")

PydanticOutputParser result:
Movie: The Matrix
Rating: 10/10
Summary: A groundbreaking sci-fi film that masterfully blends philosophy, action, and technology into a captivating narrative. The visuals and special effects were ahead of their time, creating a unique cinematic experience that continues to influence the genre.
Recommended: True


---
## Section 3: Memory Systems

Memory allows LLMs to remember previous interactions within a conversation.

In [35]:
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationSummaryBufferMemory,
    ConversationEntityMemory
)
from langchain_classic.chains import ConversationChain

### 3.1 ConversationBufferMemory
Stores the entire conversation history. Simple but can grow unbounded.

In [36]:
# ConversationBufferMemory - stores full conversation history
buffer_memory = ConversationBufferMemory(return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=buffer_memory,
    verbose=True  # Shows what's happening internally
)

# Have a conversation
print(conversation.predict(input="Hi, my name is Alice."))
print("---")
print(conversation.predict(input="What's a good restaurant in Paris?"))
print("---")
print(conversation.predict(input="What's my name?"))  # Should remember!



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Hi, my name is Alice.
AI:

> Finished chain.
Hello, Alice! It's great to meet you. I'm here to chat and help with any questions you might have. What’s on your mind today?
---


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello, Alice! It's great to meet you. I'm here to chat and

In [37]:
# Inspect the memory
print("Memory contents:")
print(buffer_memory.load_memory_variables({}))

Memory contents:
{'history': [HumanMessage(content='Hi, my name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello, Alice! It's great to meet you. I'm here to chat and help with any questions you might have. What’s on your mind today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's a good restaurant in Paris?", additional_kwargs={}, response_metadata={}), AIMessage(content="Paris is known for its incredible dining scene! One popular choice is Le Meurice, a Michelin-starred restaurant that offers a blend of classic French cuisine and modern touches. The ambiance is elegant, and the views of the Tuileries Garden are stunning. If you're looking for something more casual, you might enjoy Café de Flore, a historic café famous for its classic Parisian atmosphere and people-watching. Another great option is Le Relais de l’Entrecôte, where they serve a delicious steak-frites with a secret sauce. Ar

### 3.2 ConversationBufferWindowMemory
Keeps only the last K exchanges. Good for long conversations where only recent context matters.

In [38]:
# ConversationBufferWindowMemory - keeps last k messages
window_memory = ConversationBufferWindowMemory(k=2, return_messages=True)

conversation_window = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

# Simulate a longer conversation
print("Message 1:", conversation_window.predict(input="My favorite color is blue."))
print("\nMessage 2:", conversation_window.predict(input="I also love pizza."))
print("\nMessage 3:", conversation_window.predict(input="My dog's name is Max."))
print("\nMessage 4:", conversation_window.predict(input="What was my favorite color?"))  # Might not remember!

print("\nMemory (only last 2 exchanges):")
print(window_memory.load_memory_variables({}))

Message 1: That's great! Blue is such a calming and versatile color. It can range from the soft, serene shades of sky blue to the deep, mysterious tones of navy. Do you have a specific shade of blue that you like the most? Or perhaps a favorite item that’s blue, like clothing or decor?

Message 2: Pizza is a fantastic choice! It's one of those foods that can be customized in so many ways. Do you have a favorite type of pizza or toppings that you enjoy? For example, some people love classic pepperoni, while others might go for something adventurous like barbecue chicken or a veggie supreme. And what about the crust? Thin and crispy, or thick and chewy?

Message 3: That's a lovely name for a dog! Max is such a friendly and energetic name. What breed is he? Dogs can bring so much joy and companionship. Does he have any favorite toys or activities? For example, some dogs love to fetch, while others might prefer a good game of tug-of-war or just lounging around.

Message 4: I'm not sure wha

### 3.3 ConversationSummaryMemory
Uses an LLM to create a running summary of the conversation. Great for very long conversations.

In [39]:
# ConversationSummaryMemory - summarizes conversation
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)

conversation_summary = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=False
)

# Have a multi-turn conversation
conversation_summary.predict(input="I'm planning a trip to Japan next month.")
conversation_summary.predict(input="I want to visit Tokyo and Kyoto.")
conversation_summary.predict(input="I'm interested in temples and good food.")

print("Summary of conversation:")
print(summary_memory.load_memory_variables({}))

Summary of conversation:
{'history': [SystemMessage(content='The human is planning a trip to Japan next month and expresses a desire to visit Tokyo and Kyoto. The AI responds positively, highlighting that both cities are iconic and offer unique experiences. In Tokyo, the AI suggests visiting Shibuya Crossing, Asakusa and Senso-ji Temple, Akihabara, and either Tokyo Tower or Tokyo Skytree for panoramic views. For Kyoto, the AI recommends Kinkaku-ji (Golden Pavilion), Fushimi Inari Taisha, Arashiyama Bamboo Grove, and the Gion District, known for its geisha culture. The AI also advises trying sushi in Tokyo and kaiseki in Kyoto, and offers to provide more specific recommendations for activities, food, or travel tips. \n\nThe human expresses interest in temples and good food. The AI highlights that Japan is rich in beautiful temples and delicious food, recommending Senso-ji Temple in Tokyo for its vibrant shopping street and street food options, and Kinkaku-ji in Kyoto for its stunning sc

### 3.4 ConversationSummaryBufferMemory
Hybrid approach: keeps recent messages AND maintains a summary of older ones. Best of both worlds.

In [44]:
# ConversationSummaryBufferMemory - hybrid approach
summary_buffer_memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,  # Summarize when exceeding this limit
    return_messages=True
)

conversation_hybrid = ConversationChain(
    llm=llm,
    memory=summary_buffer_memory,
    verbose=False
)

# Long conversation
conversation_hybrid.predict(input="I'm a software engineer working on AI projects.")
conversation_hybrid.predict(input="Currently I'm building a RAG system.")
conversation_hybrid.predict(input="I use Python and LangChain.")
conversation_hybrid.predict(input="What frameworks should I also consider?")

print("Memory state (summary + recent buffer):")
print(summary_buffer_memory.load_memory_variables({}))

Memory state (summary + recent buffer):
{'history': [SystemMessage(content="The human shares that they are a software engineer working on AI projects, and the AI responds positively, expressing that AI is an exciting field with many possibilities while inquiring about the specific projects the human is working on. The human mentions they are building a Retrieval-Augmented Generation (RAG) system. The AI finds this interesting and explains that a RAG system combines retrieval and generation for more accurate responses, asking about the implementation details, such as the type of retrieval model, generator architecture, and datasets used. The human replies that they are using Python and LangChain for the project. The AI praises the choices of Python and LangChain, asking if the human is using LangChain's built-in components or integrating their own retrieval model, and inquires about the data sources and structuring of interactions. The human asks about other frameworks to consider, and 

### 3.5 Entity Memory
Extracts and tracks entities (people, places, concepts) mentioned in the conversation.

In [45]:
# ConversationEntityMemory - tracks entities
entity_memory = ConversationEntityMemory(llm=llm)

conversation_entity = ConversationChain(
    llm=llm,
    memory=entity_memory,
    verbose=False
)

# Conversation with entities
conversation_entity.predict(input="John works at Google as a machine learning engineer.")
conversation_entity.predict(input="His colleague Sarah specializes in NLP.")
conversation_entity.predict(input="They're building a chatbot together.")

print("Tracked entities:")
print(entity_memory.entity_store.store)

/var/folders/68/r5qhsk790gnct9zrkl0zp9pw0000gn/T/ipykernel_20833/2160949364.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  entity_memory = ConversationEntityMemory(llm=llm)
/opt/anaconda3/lib/python3.12/site-packages/pydantic/main.py:253: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


ValidationError: 1 validation error for ConversationChain
  Value error, Got unexpected prompt input variables. The prompt expects ['history', 'input'], but got ['entities', 'history'] as inputs from memory, and input as the normal input key. [type=value_error, input_value={'llm': ChatOpenAI(profil...={})), 'verbose': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

---
## Section 4: Chains

Chains allow you to combine multiple components (prompts, LLMs, parsers, tools) into a single pipeline.

### 4.1 LCEL (LangChain Expression Language)
Modern, recommended way to build chains using the pipe (`|`) operator.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# Simple LCEL chain: prompt | llm | parser
simple_chain = (
    ChatPromptTemplate.from_template("Tell me a fun fact about {topic}")
    | llm
    | StrOutputParser()
)

result = simple_chain.invoke({"topic": "octopuses"})
print("Simple chain result:")
print(result)

In [ ]:
# RunnablePassthrough - passes input through unchanged
# Useful when you need original input alongside processed data

chain_with_passthrough = RunnableParallel(
    original_topic=RunnablePassthrough(),
    fun_fact=ChatPromptTemplate.from_template("Tell me one fun fact about {topic}") | llm | StrOutputParser()
)

result = chain_with_passthrough.invoke({"topic": "Python"})
print("With passthrough:")
print(f"Original: {result['original_topic']}")
print(f"Fun fact: {result['fun_fact']}")

In [ ]:
# RunnableParallel - run multiple chains in parallel
parallel_chain = RunnableParallel(
    summary=ChatPromptTemplate.from_template("Summarize {topic} in one sentence") | llm | StrOutputParser(),
    key_points=ChatPromptTemplate.from_template("List 3 key points about {topic}") | llm | StrOutputParser(),
    use_cases=ChatPromptTemplate.from_template("Give 2 practical use cases for {topic}") | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "Docker containers"})
print("Parallel execution results:")
for key, value in result.items():
    print(f"\n{key.upper()}:")
    print(value)

In [ ]:
# RunnableLambda - use custom functions in chains
def format_output(data: dict) -> str:
    """Custom function to format the output."""
    return f"""
=== TOPIC ANALYSIS ===

SUMMARY:
{data['summary']}

KEY POINTS:
{data['key_points']}

USE CASES:
{data['use_cases']}
=====================
"""

full_chain = parallel_chain | RunnableLambda(format_output)

result = full_chain.invoke({"topic": "REST APIs"})
print(result)

### 4.2 Sequential Chains
Output of one step becomes input to the next.

In [ ]:
# Sequential chain: research -> outline -> write

# Step 1: Generate research notes
research_prompt = ChatPromptTemplate.from_template(
    "Research the topic '{topic}' and provide 3 key insights. Be concise."
)

# Step 2: Create outline based on research
outline_prompt = ChatPromptTemplate.from_template(
    """Based on these research notes:
{research}

Create a brief outline for a blog post."""
)

# Step 3: Write introduction based on outline
intro_prompt = ChatPromptTemplate.from_template(
    """Based on this outline:
{outline}

Write a compelling introduction paragraph (2-3 sentences)."""
)

# Build the sequential chain
sequential_chain = (
    {"topic": RunnablePassthrough()}
    | RunnableParallel(
        topic=lambda x: x["topic"],
        research=research_prompt | llm | StrOutputParser()
    )
    | RunnableParallel(
        research=lambda x: x["research"],
        outline=outline_prompt | llm | StrOutputParser()
    )
    | RunnableParallel(
        outline=lambda x: x["outline"],
        introduction=intro_prompt | llm | StrOutputParser()
    )
)

result = sequential_chain.invoke({"topic": "sustainable energy"})
print("OUTLINE:")
print(result["outline"])
print("\nINTRODUCTION:")
print(result["introduction"])

### 4.3 Router Chain
Routes inputs to different chains based on conditions.

In [ ]:
from langchain_core.runnables import RunnableBranch

# Different prompts for different types of questions
math_prompt = ChatPromptTemplate.from_template(
    "You are a math tutor. Solve this problem step by step: {question}"
)

science_prompt = ChatPromptTemplate.from_template(
    "You are a science expert. Explain this concept clearly: {question}"
)

history_prompt = ChatPromptTemplate.from_template(
    "You are a history professor. Provide context and details about: {question}"
)

general_prompt = ChatPromptTemplate.from_template(
    "Answer this question helpfully: {question}"
)

# Classification chain to determine question type
classify_prompt = ChatPromptTemplate.from_template(
    """Classify this question into one category: math, science, history, or general.
Question: {question}
Respond with ONLY the category name, nothing else."""
)

def route_question(info):
    """Route to appropriate chain based on classification."""
    category = info["category"].lower().strip()
    question = info["question"]
    
    if "math" in category:
        return math_prompt.format(question=question)
    elif "science" in category:
        return science_prompt.format(question=question)
    elif "history" in category:
        return history_prompt.format(question=question)
    else:
        return general_prompt.format(question=question)

# Full routing chain
router_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        category=classify_prompt | llm | StrOutputParser()
    )
    | RunnableLambda(route_question)
    | llm
    | StrOutputParser()
)

# Test with different types of questions
questions = [
    "What is 15% of 240?",
    "How does photosynthesis work?",
    "Who was Cleopatra?"
]

for q in questions:
    print(f"Q: {q}")
    result = router_chain.invoke(q)
    print(f"A: {result}\n")